# 03. RetinaNet — навчання (one-stage, anchor-based)

Аналіз якорів — у `03_retinanet.ipynb`. Тут навчання за протоколом `02_faster_rcnn` (torchvision, ваги COCO, 960 px, batch 4, SGD, hflip, AMP, seed 42, 300 рамок), один ноутбук на всі запуски.

| Запуск | Що змінено | Тип |
|---|---|---|
| `baseline` | стандартні якорі torchvision (32…812 px) | — |
| `exp1_anchors` | розміри якорів ×0,5 (16…406 px): покриття об'єктів 50% → 87% у `03_retinanet` | гіперпараметр |
| `exp2_giou` | втрата рамок L1 → GIoU, поверх кращого з `baseline` / `exp1_anchors` | функція втрат |

In [1]:
EXPERIMENT = "baseline"
SMOKE_TEST = False

In [2]:
# Parameters
EXPERIMENT = "exp1_anchors"


In [3]:
EXPERIMENTS = {
    "baseline": {"anchor_factor": 1.0, "box_loss": "l1"},
    "exp1_anchors": {"anchor_factor": 0.5, "box_loss": "l1"},
    "exp2_giou": {"anchor_factor": "best", "box_loss": "giou"},
}

EPOCHS = 6
BATCH = 4
LR = 0.0025
IMG_SIZE = 960
MAX_DETS = 300
VAL_STEP = 5
SEED = 42

## 1. Шляхи, бібліотеки, відтворюваність

In [4]:
import json
import math
import random
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.io import ImageReadMode, read_image
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.anchor_utils import AnchorGenerator

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    sys.path.append(str(next(Path("/kaggle/input").glob("**/hw2_eval.py")).parent))
    sys.path.append(str(next(Path("/kaggle/input").glob("**/hw2_ultra.py")).parent))
import hw2_eval as E
import hw2_ultra as U

DATA, PREPARED, RUNS = U.paths()
SETUP = dict(EXPERIMENTS[EXPERIMENT])
if SETUP["anchor_factor"] == "best":
    SETUP["anchor_factor"] = EXPERIMENTS[U.best_run(RUNS / "retinanet", ["baseline", "exp1_anchors"])]["anchor_factor"]
OUT = RUNS / "retinanet" / (f"smoke_{EXPERIMENT}" if SMOKE_TEST else EXPERIMENT)
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = U.gpu_name()
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(GPU, SETUP, OUT, sep="\n")

NVIDIA GeForce RTX 4070 Laptop GPU
{'anchor_factor': 0.5, 'box_loss': 'l1'}
/home/marta/Desktop/ucu/year3/AI/hw2_runs/retinanet/exp1_anchors


## 2. Дані

Те саме, що в `02_faster_rcnn`: train — підготовлений набір, val і test — оригінальні кадри, рамки в `[x1, y1, x2, y2]`, класи 1–8, hflip з імовірністю 0,5.

In [5]:
class FramesDataset(Dataset):
    def __init__(self, records, train=False):
        self.records = records
        self.train = train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        path, image_id, boxes, labels = self.records[i]
        image = read_image(str(path), ImageReadMode.RGB).float() / 255
        boxes = boxes.clone()
        if self.train and torch.rand(1).item() < 0.5:
            image = image.flip(-1)
            boxes[:, [0, 2]] = image.shape[-1] - boxes[:, [2, 0]]
        return image, {"boxes": boxes, "labels": labels, "image_id": image_id}


def coco_records(coco, path_of):
    by_image = defaultdict(list)
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        if a["iscrowd"] == 0 and w >= 1 and h >= 1:
            by_image[a["image_id"]].append((x, y, x + w, y + h, a["category_id"]))
    records = []
    for info in coco["images"]:
        rows = torch.tensor(by_image[info["id"]], dtype=torch.float32).reshape(-1, 5)
        records.append((path_of(info), info["id"], rows[:, :4], rows[:, 4].long()))
    return records


def collate(batch):
    return tuple(zip(*batch))


def make_loader(records, train):
    return DataLoader(FramesDataset(records, train), batch_size=BATCH, shuffle=train, collate_fn=collate,
                      num_workers=12, pin_memory=DEVICE.type == "cuda",
                      generator=torch.Generator().manual_seed(SEED))

In [6]:
val_json = DATA / "val" / "annotations_coco.json"
train_coco = json.loads((PREPARED / "train_coco.json").read_text())
val_coco = json.loads(val_json.read_text())


def val_path(info):
    p = Path(info["file_name"])
    return DATA / "val" / "images" / p.parent.name / p.name


train_records = coco_records(train_coco, lambda info: PREPARED / info["file_name"])
val_records = coco_records(val_coco, val_path)

val_frames = pd.DataFrame(val_coco["images"])
val_frames["pos"] = val_frames.groupby("video")["frame_id"].rank(method="first").astype(int) - 1
val_sub_ids = set(val_frames.loc[val_frames["pos"] % VAL_STEP == 0, "id"])
val_sub_records = [r for r in val_records if r[1] in val_sub_ids]

test_frames = pd.read_csv(DATA / "test_images.csv")
test_records = [(DATA / f, i, torch.zeros((0, 4)), torch.zeros(0, dtype=torch.long))
                for f, i in zip(test_frames["file_name"], test_frames["image_id"])]

if SMOKE_TEST:
    EPOCHS = 1
    train_records, val_records, test_records = train_records[:8], val_records[:4], test_records[:4]
    val_sub_records = val_records
    val_sub_ids = {r[1] for r in val_records}

pd.Series({"train": len(train_records), "val (кожна епоха)": len(val_sub_records),
           "val (повний)": len(val_records), "test": len(test_records)})

train                4868
val (кожна епоха)     571
val (повний)         2846
test                 3811
dtype: int64

## 3. Модель

torchvision RetinaNet ResNet-50 FPN з вагами COCO. Замінюємо лише останній шар класифікації на 9 виходів (8 класів + невикористаний 0), ініціалізація як у torchvision (`prior_probability = 0,01`), решта голови лишається попередньо навченою. Кількість якорів у точці та сама (3 розміри × 3 форми), тому голова регресії підходить і для зменшених якорів.

In [7]:
NUM_CLASSES = len(train_coco["categories"]) + 1


def anchor_generator(factor):
    sizes = tuple(tuple(int(x * factor * 2 ** (k / 3)) for k in range(3)) for x in (32, 64, 128, 256, 512))
    return AnchorGenerator(sizes, ((0.5, 1.0, 2.0),) * 5)


def build_model():
    model = retinanet_resnet50_fpn(weights="DEFAULT", min_size=IMG_SIZE, max_size=IMG_SIZE, detections_per_img=MAX_DETS)
    model.anchor_generator = anchor_generator(SETUP["anchor_factor"])
    head = model.head.classification_head
    head.cls_logits = nn.Conv2d(256, head.num_anchors * NUM_CLASSES, kernel_size=3, padding=1)
    nn.init.normal_(head.cls_logits.weight, std=0.01)
    nn.init.constant_(head.cls_logits.bias, -math.log((1 - 0.01) / 0.01))
    head.num_classes = NUM_CLASSES
    model.head.regression_head._loss_type = SETUP["box_loss"]
    return model.to(DEVICE)


pd.Series({f"P{level + 3}": sizes for level, sizes in enumerate(anchor_generator(SETUP["anchor_factor"]).sizes)})

P3       (16, 20, 25)
P4       (32, 40, 50)
P5      (64, 80, 101)
P6    (128, 161, 203)
P7    (256, 322, 406)
dtype: object

## 4. Навчання і передбачення

Як у `02_faster_rcnn`: SGD (momentum 0,9, wd 1e-4), розігрів у першій епосі, AMP, знижки lr ×0,1 на 16/26 і 22/26 епох. lr 0,01 для batch 16 з `references/detection` для RetinaNet → 0,0025 для batch 4.

In [8]:
def to_device(targets):
    return [{k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in t.items()} for t in targets]


def train_one_epoch(model, loader, optimizer, scaler, warmup):
    model.train()
    total = 0.0
    for images, targets in loader:
        images = [im.to(DEVICE, non_blocking=True) for im in images]
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            loss = sum(model(images, to_device(targets)).values())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if warmup is not None and scaler.get_scale() >= scale_before:
            warmup.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def predict(model, records):
    model.eval()
    detections = []
    for images, targets in make_loader(records, train=False):
        outputs = model([im.to(DEVICE) for im in images])
        for target, out in zip(targets, outputs):
            detections += U.to_coco(target["image_id"], out["boxes"].cpu(), out["labels"].cpu() - 1, out["scores"].cpu())
    return detections

In [9]:
model = build_model()
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=LR, momentum=0.9, weight_decay=1e-4)
milestones = [round(EPOCHS * 16 / 26), round(EPOCHS * 22 / 26)]
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.1)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == "cuda")
train_loader = make_loader(train_records, train=True)
gt_val = E.load_gt(val_json)

history = []
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    warmup = (torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1 / 1000,
                                                total_iters=min(1000, len(train_loader) - 1))
              if epoch == 1 else None)
    loss = train_one_epoch(model, train_loader, optimizer, scaler, warmup)
    scheduler.step()
    train_minutes = (time.time() - start) / 60
    summary, _ = E.metrics(gt_val, E.load_detections(gt_val, predict(model, val_sub_records)), val_sub_ids)
    history.append({"epoch": epoch, "loss": loss, "train_min": train_minutes, "total_min": (time.time() - start) / 60,
                    "mAP": summary["mAP"], "AP50": summary["AP50"], "AP_small": summary["AP_small"]})
    print(pd.DataFrame(history).tail(1).round(4).to_string(index=False, header=epoch == 1))

pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
torch.save(model.state_dict(), OUT / "model.pt")

 epoch   loss  train_min  total_min    mAP   AP50  AP_small
     1 0.8593      4.548      5.053 0.1059 0.2461    0.0406


2 0.5801 4.5222 8.5977 0.1303 0.2947 0.051


3 0.5172 4.5137 5.1431 0.1414 0.3154 0.0551


4 0.4756 4.5195 8.0767 0.1416 0.3119 0.0538


5 0.4312 4.5174 5.7786 0.1452 0.3144 0.0539


6 0.4237 4.5101 6.8862 0.1444 0.3138 0.0532


## 5. Повний val, швидкість, сабміт

In [10]:
@torch.no_grad()
def ms_per_image(model, records, n=100):
    model.eval()
    times = []
    for path, *_ in records[:n]:
        image = (read_image(str(path), ImageReadMode.RGB).float() / 255).to(DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        model([image])
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    return 1000 * float(np.median(times[5:] or times))


val_detections = predict(model, val_records)
summary, per_class, clips = E.report(gt_val, val_detections)
speed = ms_per_image(model, val_sub_records)

test_detections = predict(model, test_records)
E.write_submission(test_detections, DATA / "test_images.csv", OUT / "submission.csv")

config = {"experiment": EXPERIMENT, **SETUP, "gpu": GPU, "epochs": EPOCHS, "batch": BATCH, "lr": LR,
          "img_size": IMG_SIZE, "max_dets": MAX_DETS, "seed": SEED}
results = {"config": config, "val": summary.to_dict(), "per_class": per_class.to_dict(),
           "clips": clips.to_dict(), "ms_per_image": speed}
(OUT / "metrics.json").write_text(json.dumps(results, indent=1))
(OUT / "val_predictions.json").write_text(json.dumps(val_detections))
(OUT / "test_predictions.json").write_text(json.dumps(test_detections))

print(f"Швидкість: {speed:.1f} мс/кадр")
display(summary.round(4), per_class.round(4), clips.round(4))

Швидкість: 28.5 мс/кадр


mAP              0.1432
AP50             0.3125
AP75             0.1115
AP_small         0.0519
AP_medium        0.1249
AP_large         0.2790
mAP_test_like    0.1530
dtype: float64

person      0.1960
bicycle     0.1205
car         0.3262
van         0.1390
truck       0.1552
tricycle    0.1083
bus         0.0271
motor       0.0736
dtype: float64

uav0000086_00000_v    0.1519
uav0000117_02622_v    0.1812
uav0000137_00458_v    0.2057
uav0000182_00000_v    0.0968
uav0000268_05773_v    0.0813
uav0000305_00000_v    0.2394
uav0000339_00001_v    0.1697
dtype: float64

## 6. Дисбаланс якорів без етапу пропозицій

Для кожного кадру val (кожен 5-й) рахуємо, скільки якорів стають позитивами (IoU ≥ 0,5), негативами (IoU < 0,4) і скільки ігноруються, за правилом `Matcher(0.5, 0.4, allow_low_quality_matches=True)` з torchvision RetinaNet. Two-stage детектор після RPN бачить ~512 вибраних RoI на кадр, а одноетапний — усі якорі. Саме цей дисбаланс компенсує focal loss.

In [11]:
from torchvision.ops import box_iou


@torch.no_grad()
def anchor_balance(model, records):
    model.eval()
    rows = []
    for images, targets in make_loader(records, train=False):
        images = [im.to(DEVICE) for im in images]
        image_list, scaled = model.transform(images, to_device(targets))
        features = list(model.backbone(image_list.tensors).values())
        anchors = model.anchor_generator(image_list, features)
        for a, t in zip(anchors, scaled):
            if len(t["boxes"]) == 0:
                continue
            best = box_iou(t["boxes"], a).max(dim=0).values
            rows.append({"anchors": len(a), "positive": int((best >= 0.5).sum()), "negative": int((best < 0.4).sum()),
                         "objects": len(t["boxes"])})
    return pd.DataFrame(rows)


balance = anchor_balance(model, val_sub_records)
balance["neg_per_pos"] = balance["negative"] / balance["positive"].clip(lower=1)
balance.to_csv(OUT / "anchor_balance.csv", index=False)
balance.median().round(1)

anchors        97965.0
positive         221.0
negative       97329.0
objects           37.0
neg_per_pos      440.4
dtype: float64

## Джерела

**Бібліотеки**
- PyTorch, torchvision — RetinaNet ResNet-50 FPN з вагами COCO (`retinanet_resnet50_fpn(weights="DEFAULT")`), `AnchorGenerator`, `box_iou`: https://docs.pytorch.org/vision/main/models.html
- faster-coco-eval через спільний `hw2_eval.py`.

**Код, взятий або адаптований**
- Датасет, цикл навчання, передбачення, швидкість — з нашого `02_faster_rcnn.ipynb` (там вказані джерела torchvision).
- Рецепт RetinaNet: lr 0,01 для batch 16, 26 епох зі знижками на 16 і 22 — `references/detection/README.md`: https://github.com/pytorch/vision/tree/main/references/detection
- Розміри якорів `(x, x·2^(1/3), x·2^(2/3))` і форми (0,5; 1; 2) — `torchvision/models/detection/retinanet.py::_default_anchorgen`.
- Ініціалізація останнього шару (`normal(std=0.01)`, `bias = −log((1 − π) / π)`, π = 0,01) — `RetinaNetClassificationHead.__init__`, там само; RetinaNet, arXiv:1708.02002, розділ 4.1.
- GIoU-втрата рамок — вбудований варіант `_loss_type = "giou"` у `RetinaNetRegressionHead` (`torchvision/models/detection/_utils.py::_box_loss`).
- Пороги якорів 0,5 / 0,4 у розділі 6 — `RetinaNet(fg_iou_thresh=0.5, bg_iou_thresh=0.4)`.

**Статті**
- RetinaNet / Focal Loss — Lin et al., 2017, arXiv:1708.02002.
- GIoU — Rezatofighi et al., 2019, arXiv:1902.09630.
- YOLOv2 (k-means якорів, порівняння в `03_retinanet`) — arXiv:1612.08242.

**Наші рішення:** множник якорів 0,5 (з аналізу покриття в `03_retinanet`), 6 епох замість 12 (пік val mAP Faster R-CNN був на 2–3-й епосі) зі знижками lr у тих самих пропорціях, перерахунок lr під batch 4, підрахунок балансу якорів.